# Tutorial 03: Running a magneto-rotational and detection simulation

We can perform the simulation of a Galactic population of neutron stars in different ways.

In this tutorial, we will focus on simulating the magneto-rotational properties of neutron stars and how they are detected.

We again consider the following three surveys:

1. the Parks Multibeam Pulsar Survey (PMPS; [Manchester et al. 2001](https://ui.adsabs.harvard.edu/abs/2001MNRAS.328...17M/abstract), [Lorimer et al. 2006](https://ui.adsabs.harvard.edu/abs/2006MNRAS.372..777L/abstract)),
2. the Swinburne Parkes Multibeam Pulsar Survey (SMPS; [Edwards et al. 2001](https://ui.adsabs.harvard.edu/abs/2001MNRAS.326..358E/abstract), [Jacoby et al. 2009](https://ui.adsabs.harvard.edu/abs/2009ApJ...699.2009J/abstract)),
3. the mid- and low-latitude High Time Resolution Universe survey (HTRU; [Keith et al. 2010](https://ui.adsabs.harvard.edu/abs/2010MNRAS.409..619K/abstract)).

Using the module `mlpoppyns/simulator/simulate_population_magrot_det.py`, we can select stars from a dynamically evolved database according to the sky coverage of a given survey, evolve their properties in time and finally establish if these sources are detected by a given survey or not.

To do so, we execute the command
```
python mlpoppyns/simulator/simulate_population_magrot_det.py --dyn_data dyn_database --save_dir output/sim_magrot_det
```

To create a dynamical database, we can either use the notebook `tutorials/tutorial_notebooks/02_simulator_dyn_tutorial.ipynb` or alternatively the dynamical database already available in the directory `data/example_simulation_dyn`. Note that the dynamical database has to be sufficiently large (see below) for the following example to work.

As for the other simulator modes, to change the initial parameters, the user can directly modify the simulator configuration in `mlpoppyns/simulator/config_simulator.py` or alternatively parse a JSON file containing custom parameters for the simulation.

In this simulation mode, new stars are sampled from the database in batches and evolved until the desired 
number of detected sources is reached. Note that we can specify the number of pulsars that we want to detect
for each survey in the configuration file. For example, we might want to match the number of detected 
pulsars in a given survey in the [ATNF Pulsar Catalogue](https://www.atnf.csiro.au/research/pulsar/psrcat/). 
To this end, the script performs the following steps in a loop until a certain number of detections for each 
survey is reached:

1. We first randomly sample batches of pulsars from our dynamical database. The batch size is set to 100,000 and has 
    been optimized to match radio pulsar detections with Murriyang, the Parkes Radio Telescope. Note that the dynamical 
    database should be significantly bigger than this batch size, i.e., it should contain at least 100 times the batch 
    size. Simulating neutron stars in batches helps to speed up the simulation by evolving an array of pulsars
    simultaneously.

2. Next, we select pulsars that fall into the sky coverage of the surveys and are not further away than 35 kpc from 
    the Sun. For those objects, we then evolve the magnetic fields, spin periods and inclination angles. This 
    pre-selection allows us to not waste computational resources on pulsars that have no chance to be detected.

3. Then, we model the emission geometry and only select those pulsars that are beamed towards the Earth. A luminosity is
    associated with these pulsars and their flux is computed.

4. Finally, we apply the surveys' sensitivity thresholds to select those pulsars that are detected according to the 
    limiting flux of each survey.

A posteriori, this simulation mode allows us to determine the birth rate of our synthetic neutron star population by 
looking at the total number of neutron stars created over a specified evolution time to reach the desired number of 
detections.

To avoid wasting computational resources on unrealistic parameter regimes, we do not evolve populations whose birth rate exceeds a limiting value of 5 neutron stars per century, which is unrealistic given our current understanding of neutron star formation. In these cases, the simulation is stopped and a flag that warns about the excess in the birth rate is saved into the output configuration file.

Overall, in this mode, the output of the simulation consists of separate files containing the properties of the 
detected pulsars for each survey, a `profile.json` file and a `configuration.json` file containing the entire set of 
parameters used to simulate the magneto-rotational evolution and the detection models.

For an application of this simulation mode and the various physical models implemented see [Graber et al. (2024)](https://ui.adsabs.harvard.edu/abs/2024ApJ...968...16G/abstract).

In [ ]:
import argparse
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg
from mlpoppyns.simulator.simulate_population_magrot_det import (
    simulate_population,
)

WARNING: If you see a warning here, make sure to set the path to the repository in the simulator configuration file. Otherwise, the examples below will not run.

## Setting up and running the magneto-rotational simulation

We can adjust several parameters in the imported configuration file.
For example, we can change the following:
1. `B_initial_log10_mean` and `B_initial_log10_sigma`: the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial magnetic field.
2. `P_initial_log10_mean` and `P_initial_log10_sigma`: the mean and standard deviation of the Gaussian distribution for the $\log_{10}$ of the initial spin period.
3. `a_late`: the power-law index for the late time decay of the magnetic field after $\sim 10^6$ yr.

NOTE: While we could, in principle, adjust the number of neutron stars and their maximum age here as we did in Tutorials 01 and 02, this would not affect the following simulation as the neutron stars will be randomly picked from the dynamical database and the `t_age_max` will be set to the one of the dynamical database. In general, one has to make sure that the dynamical database is sufficiently large for the detection loop to work (see notes above). We, thus, recommend that `NS_number` and `t_age_max` are not changed here.

We will define the following simulation parameters:

In [ ]:
cfg["B_initial_log10_mean"] = 13.1
cfg["B_initial_log10_sigma"] = 0.45
cfg["P_initial_log10_mean"] = -1.0
cfg["P_initial_log10_sigma"] = 0.38
cfg["a_late"] = -1.80

Alternatively, we can directly change the parameters in the `parameter_override.json` file in the `tutorials/tutorial_notebooks` folder and pass it to the simulator. Note that we will not use this in the example below, however.

In [ ]:
override_dir = "parameter_override.json"

We next specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_magrot_det"

We can now run the simulation by calling the `simulate_population` function from the `mlpoppyns.simulator.simulate_population_magrot_det` module with our specific parameter choices and the database stored in the `/data/example_simulation_dyn` directory.

WARNING: If a `FileNotFound` error appears, remember to check the path to the repository in the `mlpoppyns/simulator/config_simulator.json` configuration file as outlined in the GitHub `README.md` and the `Getting started` page of our documentation.

In [ ]:
simulation_args = argparse.Namespace(
    dyn_data="../../data/example_simulation_dyn",
    save_dir=output_dir,
    parameter_override=None,
)
simulate_population(simulation_args)

## Reading the simulation results

We first read in our compressed `.pkl` files.

In [ ]:
data_PMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_PMPS_results.pkl.gz"),
    compression="gzip",
)
data_PMPS.columns

In [ ]:
data_SMPS = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_SMPS_results.pkl.gz"),
    compression="gzip",
)
data_SMPS.columns

In [ ]:
data_HTRU_low_mid = pd.read_pickle(
    pathlib.Path().joinpath(output_dir, "survey_HTRU_low_mid_results.pkl.gz"),
    compression="gzip",
)
data_HTRU_low_mid.columns

Extracting the detection numbers and corresponding birth rates.

In [ ]:
print(f"Number of pulsars detected by PMPS: {len(data_PMPS)}")
print(f"Number of pulsars detected by SMPS: {len(data_SMPS)}")
print(
    f"Number of pulsars detected by HTRU low and mid latitude: {len(data_HTRU_low_mid)}"
)

config_filename = pathlib.Path().joinpath(output_dir, "configuration.json")
with config_filename.open("rt") as handle:
    config = json.load(handle, object_hook=collections.OrderedDict)

br_PMPS = config["birth_rate_PMPS_at_match"]
br_SMPS = config["birth_rate_SMPS_at_match"]
br_HTRU_low_mid = config["birth_rate_HTRU_low_mid_at_match"]

print(f"Pulsar birth rate predicted by PMPS: {br_PMPS}")
print(f"Pulsar birth rate predicted by SMPS: {br_SMPS}")
print(
    f"Pulsar birth rate predicted by HTRU low and mid latitude: {br_HTRU_low_mid}"
)

Extracting the parameters of detected pulsars in the individual surveys.

In [ ]:
ra_pk_sim = data_PMPS["ra"]["[deg]"].to_numpy()
dec_pk_sim = data_PMPS["dec"]["[deg]"].to_numpy()
l_pk_sim = data_PMPS["l"]["[deg]"].to_numpy()
b_pk_sim = data_PMPS["b"]["[deg]"].to_numpy()
pmra_pk_sim = data_PMPS["pm_ra"]["[mas yr^-1]"].to_numpy()
pmdec_pk_sim = data_PMPS["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_pk_sim = data_PMPS["DM"]["[pc cm^-3]"].to_numpy()
dist_pk_sim = data_PMPS["dist"]["[kpc]"].to_numpy()
P_pk_sim = data_PMPS["P"]["[s]"].to_numpy()
Pdot_pk_sim = data_PMPS["P_dot"]["[s s^-1]"].to_numpy()
S1400_pk_sim = data_PMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_pk_sim = data_PMPS["w_eff"]["[s]"].to_numpy()

In [ ]:
ra_sw_sim = data_SMPS["ra"]["[deg]"].to_numpy()
dec_sw_sim = data_SMPS["dec"]["[deg]"].to_numpy()
l_sw_sim = data_SMPS["l"]["[deg]"].to_numpy()
b_sw_sim = data_SMPS["b"]["[deg]"].to_numpy()
pmra_sw_sim = data_SMPS["pm_ra"]["[mas yr^-1]"].to_numpy()
pmdec_sw_sim = data_SMPS["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_sw_sim = data_SMPS["DM"]["[pc cm^-3]"].to_numpy()
dist_sw_sim = data_SMPS["dist"]["[kpc]"].to_numpy()
P_sw_sim = data_SMPS["P"]["[s]"].to_numpy()
Pdot_sw_sim = data_SMPS["P_dot"]["[s s^-1]"].to_numpy()
S1400_sw_sim = data_SMPS["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_sw_sim = data_SMPS["w_eff"]["[s]"].to_numpy()

In [ ]:
ra_htru_sim = data_HTRU_low_mid["ra"]["[deg]"].to_numpy()
dec_htru_sim = data_HTRU_low_mid["dec"]["[deg]"].to_numpy()
l_htru_sim = data_HTRU_low_mid["l"]["[deg]"].to_numpy()
b_htru_sim = data_HTRU_low_mid["b"]["[deg]"].to_numpy()
pmra_htru_sim = data_HTRU_low_mid["pm_ra"]["[mas yr^-1]"].to_numpy()
pmdec_htru_sim = data_HTRU_low_mid["pm_dec"]["[mas yr^-1]"].to_numpy()
DM_htru_sim = data_HTRU_low_mid["DM"]["[pc cm^-3]"].to_numpy()
dist_htru_sim = data_HTRU_low_mid["dist"]["[kpc]"].to_numpy()
P_htru_sim = data_HTRU_low_mid["P"]["[s]"].to_numpy()
Pdot_htru_sim = data_HTRU_low_mid["P_dot"]["[s s^-1]"].to_numpy()
S1400_htru_sim = data_HTRU_low_mid["S_radio_obs_mean"]["[Jy]"].to_numpy()
w_eff_htru_sim = data_HTRU_low_mid["w_eff"]["[s]"].to_numpy()

## Plotting the simulation results

As a first diagnostic, we plot our population and corresponding detection in Galactic longitude and latitude.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.plot(
    l_pk_sim,
    b_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=8,
    alpha=1,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    l_sw_sim,
    b_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    l_htru_sim,
    b_htru_sim,
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=6,
    alpha=1,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.plot(0.0, 0.0, marker="*", color="tab:orange", markersize=20)
ax.set_xlim(-150.0, 100.0)
ax.set_ylim(-50.0, 50.0)
ax.set_xlabel("l [deg]")
ax.set_ylabel("b [deg]")
ax.legend(frameon=True, loc=0)

plt.show()

We can also visualize our population and corresponding detections in the $P-\dot{P}$ plane.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 12))

ax.plot(
    P_pk_sim,
    Pdot_pk_sim,
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=8,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated PMPS",
)
ax.plot(
    P_sw_sim,
    Pdot_sw_sim,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated SMPS",
)
ax.plot(
    P_htru_sim,
    Pdot_htru_sim,
    linestyle="None",
    marker="o",
    # fillstyle="none",
    color="tab:purple",
    markersize=6,
    alpha=1.0,
    rasterized=True,
    label=r"Simulated HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-2, 30.0)
ax.set_ylim(1.0e-19, 1.0e-11)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"$\dot{P}$ [s/s]")
ax.legend(frameon=True, loc=2)

plt.show()

Finally, we cal also plot the fluxes of our populations.

In [ ]:
S_radio_bins = np.logspace(-5, 1, 31)

fig, ax = plt.subplots(figsize=(15, 8))

ax.hist(
    S1400_pk_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Simulated PMPS",
)
ax.hist(
    S1400_sw_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Simulated SMPS",
)
ax.hist(
    S1400_htru_sim,
    bins=S_radio_bins,
    histtype="step",
    edgecolor="tab:purple",
    lw=4,
    label="Simulated HTRU",
)
plt.xlabel(r"$S_{\rm radio}$ [Jy]")
plt.ylabel(r"Number of NSs")
ax.set_xscale("log")
ax.legend(frameon=True, loc=0)

plt.show()

To compare the simulation results with the real observations in the ATNF Pulsar Catalogue in more detail use the notebook `tutorials/analysis_notebook/survey_sim_vs_obs.ipynb`.